# 01b — Replogle 2022: Dimensionality Reduction and Visualization

**Dataset:** ReplogleWeissman2022_rpe1 (scPerturb-harmonized, post-QC from 01a)  
**Accession:** GSE181897  
**Phase:** 2  
**Date:** 2026-03-11  
**Objective:** Build and compare three representations of the perturbation landscape:
1. **PCA + UMAP** — fast, linear baseline (from 01a preprocessing)
2. **scVI latent space** — deep generative model that denoises counts and learns a nonlinear embedding
3. **scVI-corrected UMAP** — UMAP computed from the scVI latent space

The notebook provides generous explanations of the scVI model architecture, training procedure,
and why a generative model outperforms PCA for perturbation data.

## Table of Contents

1. [Setup](#setup)
2. [Load QC'd AnnData from 01a](#load)
3. [PCA-based UMAP (baseline)](#pca-umap) — quick, interpretable, noise-limited
4. [scVI: Background and Model Architecture](#scvi-background)
   - 4a. Why use a generative model for perturbation data?
   - 4b. The variational autoencoder (VAE) framework
   - 4c. scVI's count model: zero-inflated negative binomial
   - 4d. The latent space and what it represents
   - 4e. Training: ELBO, KL divergence, and reconstruction loss
5. [scVI: Data Preparation](#scvi-prep) — register AnnData, set up the model
6. [scVI: Training](#scvi-train) — fit the model, monitor convergence
7. [scVI: Latent Space Extraction](#scvi-latent) — get the 10-D embedding
8. [scVI-corrected UMAP](#scvi-umap) — compute UMAP from scVI latent space
9. [Side-by-side Comparison: PCA-UMAP vs. scVI-UMAP](#comparison)
10. [Perturbation-level Visualizations](#pert-viz) — highlight specific perturbations
11. [Leiden Clustering on scVI Embedding](#leiden) — unsupervised transcriptional programs
12. [Save Results](#save)
13. [Summary](#summary)

<a id='setup'></a>
## 0. Setup

In [ ]:
import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scipy.sparse as sp
import boto3
import warnings
warnings.filterwarnings('ignore')

import scvi

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, frameon=False)
np.random.seed(42)
scvi.settings.seed = 42

from pathlib import Path
PROC_DIR    = Path('../../data/processed/scperturb')
MODEL_DIR   = Path('../../data/processed/scperturb/scvi_model_rpe1')
RESULTS     = Path('../../results')
FIG_DIR     = RESULTS / 'figures'
for d in [PROC_DIR, MODEL_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

S3_BUCKET  = 'learn-perturb-seq'
PROC_PATH  = PROC_DIR / 'ReplogleWeissman2022_rpe1_qc.h5ad'
OUT_PATH   = PROC_DIR / 'ReplogleWeissman2022_rpe1_scvi.h5ad'

N_LATENT   = 10     # scVI latent dimension (default)
N_HIDDEN   = 128    # hidden layer width
N_LAYERS   = 1      # encoder/decoder depth
MAX_EPOCHS = 200    # max training epochs (early stopping typically triggers earlier)
BATCH_SIZE = 256    # minibatch size

print(f'scvi-tools version: {scvi.__version__}')
print('Setup complete.')

<a id='load'></a>
## 1. Load QC'd AnnData from 01a

We load the processed AnnData saved by `01a_qc_preprocessing.ipynb`, which contains:
- `adata.X` — log-normalised expression (library-size normalised to 10K, log1p)
- `adata.raw` — original raw counts (needed by scVI)
- `adata.obsm['X_pca']` — 50-component PCA embedding
- `adata.obs` — all QC annotations including `pert_cell_tier` and `guide_ambiguous`
- `adata.var['highly_variable']` — the 2,000 HVGs selected in 01a

In [ ]:
if not PROC_PATH.exists():
    # Try downloading from S3
    print(f'{PROC_PATH} not found locally. Checking S3...')
    s3 = boto3.client('s3')
    try:
        s3.download_file(S3_BUCKET, f'processed/scperturb/{PROC_PATH.name}', str(PROC_PATH))
        print('Downloaded from S3.')
    except Exception as e:
        raise FileNotFoundError(
            f'QC file not found locally or in S3.\n'
            f'Run 01a_qc_preprocessing.ipynb first to generate {PROC_PATH.name}'
        )

adata = sc.read_h5ad(PROC_PATH)
print(adata)
print()
print(f'Cells: {adata.n_obs:,}   Genes: {adata.n_vars:,}')
print(f'PCA:   {"X_pca" in adata.obsm}  (shape {adata.obsm["X_pca"].shape if "X_pca" in adata.obsm else "N/A"})')
print(f'Raw:   {adata.raw is not None}')
print(f'HVGs:  {adata.var["highly_variable"].sum() if "highly_variable" in adata.var.columns else "N/A"}')

<a id='pca-umap'></a>
## 2. PCA-based UMAP (baseline)

UMAP (Uniform Manifold Approximation and Projection) is a nonlinear dimensionality
reduction algorithm that projects high-dimensional data into 2-D for visualization.
It works by:

1. Building a k-nearest-neighbour graph in the high-dimensional space (PCA here)
2. Optimizing a 2-D layout that preserves the graph's local structure

**UMAP preserves local structure (not global distances):** Cells that are neighbours in
PCA space will be neighbours in the UMAP plot. However, the absolute distances between
clusters and the relative positions of distant clusters are **not** meaningful — UMAP
can stretch, compress, and rotate global structure arbitrarily.

PCA-based UMAP is our baseline: it is fast, deterministic (given a seed), and requires
no model training. Its limitation is that PCA is a linear projection — it cannot capture
nonlinear structure in gene expression space (e.g., curved trajectories, complex
perturbation interactions).

In [ ]:
# Compute UMAP from the 50-PC embedding produced in 01a.
# The k-NN graph was already computed in 01a (stored in adata.obsp).
# If not present, recompute it.
if 'neighbors' not in adata.uns:
    sc.pp.neighbors(adata, n_neighbors=20, use_rep='X_pca')

sc.tl.umap(adata, random_state=42)
print('PCA-UMAP coordinates stored in adata.obsm["X_umap"].')
print(f'Shape: {adata.obsm["X_umap"].shape}')

# Quick overview plot coloured by perturbation status (control vs. perturbed)
adata.obs['is_control'] = (adata.obs['perturbation'] == 'control').astype(str)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: control vs. perturbed
ctrl_mask = adata.obs['is_control'] == 'True'
axes[0].scatter(
    adata.obsm['X_umap'][~ctrl_mask, 0], adata.obsm['X_umap'][~ctrl_mask, 1],
    s=1, alpha=0.1, c='steelblue', label='perturbed', rasterized=True,
)
axes[0].scatter(
    adata.obsm['X_umap'][ctrl_mask, 0], adata.obsm['X_umap'][ctrl_mask, 1],
    s=1, alpha=0.3, c='tomato', label='control', rasterized=True,
)
axes[0].set_title('PCA-UMAP: control vs. perturbed')
axes[0].legend(markerscale=5, fontsize=9)
axes[0].set_xlabel('UMAP 1')
axes[0].set_ylabel('UMAP 2')

# Right: coloured by total UMI count (technical covariate — should not dominate structure)
col_umi = 'ncounts' if 'ncounts' in adata.obs.columns else 'total_counts'
sc_im = axes[1].scatter(
    adata.obsm['X_umap'][:, 0], adata.obsm['X_umap'][:, 1],
    s=1, alpha=0.1, c=np.log1p(adata.obs[col_umi].values),
    cmap='viridis', rasterized=True,
)
plt.colorbar(sc_im, ax=axes[1], label='log1p(UMI count)')
axes[1].set_title('PCA-UMAP: coloured by UMI count')
axes[1].set_xlabel('UMAP 1')
axes[1].set_ylabel('UMAP 2')

plt.suptitle('Baseline PCA-UMAP — ReplogleWeissman2022_rpe1', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / '01b_pca_umap_overview.pdf', bbox_inches='tight')
plt.show()

<a id='scvi-background'></a>
## 3. scVI: Background and Model Architecture

### 4a. Why use a generative model for perturbation data?

PCA is a *linear* projection — it finds the directions of maximum variance under a
Gaussian assumption. Single-cell RNA-seq data violates these assumptions in several ways:

| Property | PCA assumption | scRNA-seq reality |
|----------|---------------|-------------------|
| Distribution | Gaussian | Highly zero-inflated, overdispersed (negative binomial) |
| Noise model | Homoscedastic (equal noise) | Heteroscedastic (noise scales with expression level) |
| Dimensionality | Linear subspace | Nonlinear manifold (e.g., curved differentiation trajectories) |
| Batch effects | Not modelled | Library-size variation, sequencing depth confounds |

**Why this matters for Perturb-seq specifically:**

Perturbation effects are often *subtle* — a typical CRISPRi knockdown changes the
expression of 50–200 genes by modest amounts (log2 fold-change 0.3–1.0). These signals
are easily masked by:
- **Dropout noise:** Zeros in the count matrix arise from both genuine zero expression
  *and* stochastic failure to capture low-abundance transcripts. PCA treats all zeros as
  identical — scVI models the probability that a zero is "real" vs. "technical."
- **Library-size variation:** Cells with different total UMI counts have systematically
  different raw expression profiles. PCA partially handles this via log-normalisation,
  but the correction is imperfect. scVI explicitly models library size as a latent variable.
- **Overdispersion:** Gene expression has higher variance than a Poisson distribution
  (biological variation on top of sampling noise). scVI uses a negative binomial count model
  that captures this extra variance.

By learning a proper generative model of the count data, scVI produces a **denoised
latent space** where perturbation effects are more clearly separated from technical noise.

---

### 4b. The variational autoencoder (VAE) framework

scVI is built on the **variational autoencoder** architecture, a deep generative model
that learns to encode high-dimensional observations into a compact latent representation
and decode them back.

```
         ENCODER                    LATENT                     DECODER
   (neural network)               SPACE                  (neural network)
                              ┌───────────┐
  Gene counts ──────►  μ(z)  ──►           ──────►  Gene expression
  (n_genes = 2000)     σ(z)  ──►   z ∈ R^10  ──────►  parameters
  Library size    ──────►         │           │           ──────►  Reconstructed
  (Batch label)  ──────►         └───────────┘           counts
```

**Encoder (inference network):**
- Input: raw counts for HVGs + library size + (optionally) batch label
- Output: parameters of the *approximate posterior* $q(z | x)$
  - $\mu(z)$: mean of the latent distribution
  - $\sigma(z)$: standard deviation of the latent distribution
- Architecture: fully connected layers with ReLU activations
- The encoder learns to *compress* the high-dimensional expression profile into
  a 10-dimensional summary that retains biologically relevant variation

**Latent space:**
- $z \in \mathbb{R}^{10}$ — a 10-dimensional continuous vector per cell
- Regularised by a standard Normal prior $p(z) = \mathcal{N}(0, I)$
- During training, $z$ is sampled from $q(z|x)$ using the **reparameterisation trick**:
  $z = \mu + \sigma \odot \epsilon$, where $\epsilon \sim \mathcal{N}(0, I)$
- This is the representation we extract for downstream analysis (UMAP, E-distance, clustering)

**Decoder (generative network):**
- Input: latent vector $z$ + library size $\ell$
- Output: parameters of the count distribution for each gene
- For each gene $g$, the decoder outputs:
  - $\rho_g(z)$: normalised mean expression (fraction of total counts)
  - $\theta_g$: inverse dispersion parameter (overdispersion)
  - $\pi_g(z)$: dropout probability (for zero-inflated variant)
- Reconstructed mean: $\mu_g = \ell \cdot \rho_g(z)$

---

### 4c. scVI's count model: zero-inflated negative binomial (ZINB)

The generative model for observed counts $x_g$ (gene $g$ in a cell) is:

$$x_g \sim \begin{cases} 0 & \text{with probability } \pi_g \quad \text{(technical dropout)} \\ \text{NB}(\mu_g, \theta_g) & \text{with probability } 1 - \pi_g \end{cases}$$

where the negative binomial (NB) distribution has:
- Mean $\mu_g = \ell \cdot \rho_g(z)$ — the cell's library size times the decoded expression fraction
- Inverse dispersion $\theta_g$ — controls how much extra variance there is beyond Poisson
  - $\theta_g \to \infty$: NB approaches Poisson (no overdispersion)
  - $\theta_g \to 0$: highly overdispersed (large variance)

**The negative binomial variance:** $\text{Var}(x_g) = \mu_g + \frac{\mu_g^2}{\theta_g}$

This is always ≥ the Poisson variance ($\mu_g$), with the extra term $\mu_g^2 / \theta_g$
capturing biological cell-to-cell variation.

**Why zero-inflated?**
In scRNA-seq, zeros arise from two distinct sources:
1. **Biological zeros:** The gene is genuinely not expressed in this cell
2. **Technical zeros (dropout):** The gene is expressed but its mRNA was not captured
   during library preparation (stochastic sampling at low expression levels)

The ZINB model separates these two sources: the $\pi_g$ term models the probability of
technical dropout, while the NB component models the expression level conditional on
the transcript being captured. This denoising is a key advantage over PCA, which cannot
distinguish the two types of zeros.

**Note:** Recent scVI versions default to the standard NB model (not ZINB), as the
zero-inflation component is often unnecessary with modern 10x Chromium data where
dropout rates are lower. We use the default NB model here unless the data shows strong
zero-inflation.

---

### 4d. The latent space — what it represents

After training, each cell is encoded as a 10-dimensional vector $z_i = \mu(x_i)$
(the mean of the approximate posterior). This latent vector captures:

- **Biological variation:** Perturbation effects, cell cycle state, stress response
- **NOT technical variation:** Library size and sequencing depth are modelled as separate
  latent variables and do not contaminate $z$

The latent space is the *denoised biological state* of each cell. Distances in this space
are more biologically meaningful than distances in PCA space because:
1. The encoder has learned which expression patterns are real vs. noise
2. Library-size effects have been factored out
3. The negative binomial count model respects the data's true statistical properties

**Dimensionality:** 10 latent dimensions is the default and works well for most datasets.
Unlike PCA's 50 components (which include noise-dominated late PCs), all 10 scVI dimensions
are informative — the KL regularisation ensures that uninformative dimensions collapse to
the prior and carry no information.

---

### 4e. Training: ELBO, KL divergence, and reconstruction loss

scVI is trained by maximising the **Evidence Lower Bound (ELBO)** on the marginal
log-likelihood of the data:

$$\text{ELBO} = \underbrace{\mathbb{E}_{q(z|x)}[\log p(x|z)]}_{\text{reconstruction loss}} - \underbrace{\text{KL}[q(z|x) \| p(z)]}_{\text{regularisation}}$$

**Term 1 — Reconstruction loss:** How well can the decoder reconstruct the observed counts
from the latent representation? This is the negative binomial log-likelihood evaluated at
the true counts. Maximising this term pushes the model to preserve biological signal in $z$.

**Term 2 — KL divergence:** How far is the approximate posterior $q(z|x)$ from the prior
$p(z) = \mathcal{N}(0, I)$? This regularisation prevents the model from memorising each
cell (overfitting) and encourages a smooth, structured latent space.

**The tension between the two terms is what makes the model useful:**
- If we only optimised reconstruction, every cell would get a unique $z$ → no generalisation
- If we only optimised KL, all cells would map to $z = 0$ → no biological information
- The ELBO balances these: the model must find a compact representation that is good enough
  to reconstruct the counts while remaining smooth and regular

**Training procedure:**
- Minibatch stochastic gradient descent (Adam optimiser)
- Each epoch processes all cells in random minibatches of size 256
- Early stopping monitors the ELBO on a held-out validation set (default 10% of cells)
- Training typically converges in 50–150 epochs for datasets of this size

**When to worry about training:**
- **Training loss still decreasing at max_epochs:** Increase max_epochs or reduce batch_size
- **Validation loss diverging from training loss:** Overfitting — reduce model capacity
  (n_hidden, n_layers) or increase KL weight
- **Very fast convergence (< 10 epochs):** Model may be too small to capture the data's
  complexity — increase n_hidden or n_layers

<a id='scvi-prep'></a>
## 4. scVI: Data Preparation

scVI requires **raw counts** (not log-normalised values) because the model has its own
internal normalisation via the library-size latent variable. We recover raw counts from
`adata.raw` and subset to HVGs.

The `scvi.model.SCVI.setup_anndata()` call registers the AnnData with the model,
telling scVI where to find counts and (optionally) batch labels, categorical covariates,
and continuous covariates.

In [ ]:
# ---- Recover raw counts and subset to HVGs --------------------------------
# scVI expects raw integer counts as input.
# adata.raw (stored by 01a before normalisation) contains the original counts.

if adata.raw is not None:
    adata_raw = adata.raw.to_adata()
    print(f'Raw counts recovered from adata.raw: {adata_raw.shape}')
else:
    raise ValueError(
        'adata.raw is None — raw counts are required for scVI.\n'
        'Re-run 01a with adata.raw = adata.copy() before normalisation.'
    )

# Subset raw counts to the HVGs selected in 01a
hvg_genes = adata.var_names[adata.var['highly_variable']]
hvg_genes_in_raw = hvg_genes[hvg_genes.isin(adata_raw.var_names)]
adata_scvi = adata_raw[:, hvg_genes_in_raw].copy()

# Verify these are raw integer counts
X_sample = adata_scvi.X[:100, :100]
if sp.issparse(X_sample):
    X_sample = X_sample.toarray()
print(f'\nscVI input AnnData: {adata_scvi.shape}')
print(f'Value range (sample): [{X_sample.min():.0f}, {X_sample.max():.0f}]')
print(f'Are values integers?  {np.allclose(X_sample, X_sample.astype(int))}')

# ---- Register AnnData with scVI -------------------------------------------
# setup_anndata tells scVI:
#   - Where the count matrix is (adata.X by default)
#   - What batch/covariate columns to use (none here — single batch)
# If the dataset had multiple batches (e.g., sequencing lanes), we would pass
# batch_key='batch' to let scVI correct for batch effects.
scvi.model.SCVI.setup_anndata(adata_scvi)
print('\nAnnData registered with scVI.')
print(f'  Registered fields: {list(adata_scvi.uns["_scvi"]["data_registry"].keys())}')

<a id='scvi-train'></a>
## 5. scVI: Training

### Model configuration

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `n_latent` | 10 | Default; captures major axes of variation without overfitting |
| `n_hidden` | 128 | Width of encoder/decoder hidden layers; 128 is standard for < 500K cells |
| `n_layers` | 1 | Single hidden layer; sufficient for this dataset size |
| `gene_likelihood` | `'nb'` | Negative binomial (not ZINB) — modern 10x data has low dropout |
| `max_epochs` | 200 | Upper bound; early stopping will halt training when validation loss plateaus |
| `batch_size` | 256 | Minibatch size; 256 balances gradient noise and GPU utilisation |

For genome-scale datasets with > 200K cells, training takes 10–30 minutes on a CPU
(`r7i.4xlarge`) or 2–5 minutes on a GPU (`g4dn.2xlarge`). The model weights are saved
after training for reproducibility and to avoid re-training in subsequent notebooks.

In [ ]:
# ---- Initialise the scVI model --------------------------------------------
model = scvi.model.SCVI(
    adata_scvi,
    n_latent=N_LATENT,
    n_hidden=N_HIDDEN,
    n_layers=N_LAYERS,
    gene_likelihood='nb',   # negative binomial (no zero inflation)
)
print('scVI model initialised:')
print(f'  Parameters: {sum(p.numel() for p in model.module.parameters()):,}')
print(f'  Encoder:    {adata_scvi.n_vars} genes → {N_HIDDEN} hidden → {N_LATENT} latent')
print(f'  Decoder:    {N_LATENT} latent → {N_HIDDEN} hidden → {adata_scvi.n_vars} genes')

# ---- Check for a previously trained model ----------------------------------
if MODEL_DIR.exists() and (MODEL_DIR / 'model.pt').exists():
    print(f'\nPre-trained model found at {MODEL_DIR}. Loading...')
    model = scvi.model.SCVI.load(str(MODEL_DIR), adata=adata_scvi)
    print('Model loaded successfully.')
else:
    # ---- Train the model ---------------------------------------------------
    print(f'\nTraining scVI (max_epochs={MAX_EPOCHS}, batch_size={BATCH_SIZE})...')
    model.train(
        max_epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        early_stopping=True,
        early_stopping_patience=15,      # stop if no improvement for 15 epochs
        early_stopping_monitor='elbo_validation',
        train_size=0.9,                  # 90% train, 10% validation
        check_val_every_n_epoch=1,
    )

    # ---- Save the trained model -------------------------------------------
    model.save(str(MODEL_DIR), overwrite=True)
    print(f'\nModel saved to {MODEL_DIR}')

# ---- Plot training history -------------------------------------------------
try:
    train_history = model.history
    fig, ax = plt.subplots(figsize=(8, 3.5))
    if 'elbo_train' in train_history:
        ax.plot(train_history['elbo_train'].index,
                train_history['elbo_train'].values, label='Train ELBO', color='steelblue')
    if 'elbo_validation' in train_history:
        ax.plot(train_history['elbo_validation'].index,
                train_history['elbo_validation'].values, label='Validation ELBO', color='tomato')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('ELBO')
    ax.set_title('scVI training convergence')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / '01b_scvi_training_history.pdf', bbox_inches='tight')
    plt.show()
    
    n_epochs = len(train_history.get('elbo_train', []))
    print(f'Training completed after {n_epochs} epochs.')
except Exception as e:
    print(f'Could not plot training history: {e}')

<a id='scvi-latent'></a>
## 6. scVI: Latent Space Extraction

After training, we extract the **posterior mean** $\mu(z_i)$ for each cell $i$.
This is the "best guess" for each cell's position in the 10-dimensional latent space.

Why the posterior *mean* and not a sample?
- The posterior $q(z_i | x_i) = \mathcal{N}(\mu_i, \sigma_i^2)$ is a distribution, not a point.
  Sampling from it would introduce stochastic noise into the embedding.
- The mean is deterministic and reproducible — two calls with the same data give the
  same embedding.
- For downstream analyses (UMAP, clustering, E-distance), a deterministic embedding
  is preferable.

The latent space captures the denoised biological state. We store it in
`adata.obsm['X_scVI']` alongside the PCA embedding.

In [ ]:
# ---- Extract latent representation ----------------------------------------
# get_latent_representation returns the posterior mean by default.
latent = model.get_latent_representation(adata_scvi)
print(f'scVI latent representation: {latent.shape}')
print(f'  (cells={latent.shape[0]}, latent_dims={latent.shape[1]})')

# Store in the main adata (which has the log-normalised expression)
adata.obsm['X_scVI'] = latent
print('\nLatent space stored in adata.obsm["X_scVI"].')

# ---- Quick diagnostics on the latent space --------------------------------
print(f'\nLatent space statistics:')
print(f'  Mean per dimension:  {latent.mean(axis=0).round(3)}')
print(f'  Std per dimension:   {latent.std(axis=0).round(3)}')
print(f'  Min:                 {latent.min():.3f}')
print(f'  Max:                 {latent.max():.3f}')

# Check that no dimensions have collapsed to zero variance (would indicate
# that the KL regularisation dominated and that dimension carries no info).
active_dims = (latent.std(axis=0) > 0.1).sum()
print(f'\nActive dimensions (std > 0.1): {active_dims} / {N_LATENT}')
if active_dims < N_LATENT:
    print(f'  Note: {N_LATENT - active_dims} dimension(s) near-collapsed to the prior.')
    print('  This is normal — the model uses only as many dimensions as needed.')

# ---- Pairwise correlation between latent dimensions -----------------------
# Good latent spaces have relatively uncorrelated dimensions (each captures
# a different source of variation).
latent_corr = np.corrcoef(latent.T)
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(latent_corr, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Pearson r')
ax.set_xticks(range(N_LATENT))
ax.set_yticks(range(N_LATENT))
ax.set_xticklabels([f'z{i}' for i in range(N_LATENT)], fontsize=8)
ax.set_yticklabels([f'z{i}' for i in range(N_LATENT)], fontsize=8)
ax.set_title('scVI latent dimension correlations')
plt.tight_layout()
plt.savefig(FIG_DIR / '01b_scvi_latent_correlation.pdf', bbox_inches='tight')
plt.show()

<a id='scvi-umap'></a>
## 7. scVI-corrected UMAP

Now we compute a UMAP from the scVI latent space rather than PCA. This UMAP will reflect
the denoised, nonlinear structure learned by the generative model.

**Expected improvements over PCA-UMAP:**
- Tighter, more separated clusters (less noise in the embedding)
- Perturbation effects may be more visually distinct
- Library-size gradients should be absent (scVI factors them out)
- Better separation of control cells from weakly perturbed cells

We compute a *new* k-NN graph from the scVI embedding and then run UMAP on that graph.
This is stored as `X_umap_scvi` to keep it separate from the PCA-based UMAP.

In [ ]:
# ---- Compute k-NN graph from scVI latent space ----------------------------
# We use a separate key so both the PCA- and scVI-based graphs coexist.
sc.pp.neighbors(adata, n_neighbors=20, use_rep='X_scVI', key_added='scvi_neighbors')

# ---- Compute UMAP from scVI k-NN graph ------------------------------------
sc.tl.umap(adata, neighbors_key='scvi_neighbors', random_state=42)

# Store the scVI-UMAP coordinates under a distinct key
adata.obsm['X_umap_scvi'] = adata.obsm['X_umap'].copy()
print('scVI-UMAP coordinates stored in adata.obsm["X_umap_scvi"].')

# Restore the PCA-based UMAP as the default X_umap (recompute it)
sc.tl.umap(adata, random_state=42)  # uses the original PCA-based neighbors
print('PCA-UMAP restored as default X_umap.')

<a id='comparison'></a>
## 8. Side-by-side Comparison: PCA-UMAP vs. scVI-UMAP

We compare the two embeddings across several colourings:

1. **Control vs. perturbed** — do control cells form a coherent cluster?
2. **UMI count** — is there a library-size gradient? (scVI should remove it)
3. **Perturbation cell tier** — are well-powered perturbations in different regions?
4. **Guide ambiguity** — do ambiguous cells cluster separately?

In [ ]:
# ---- Side-by-side comparison plots ----------------------------------------
# Subsample for plotting speed
N_PLOT    = min(80_000, adata.n_obs)
rng_plot  = np.random.default_rng(42)
plot_idx  = rng_plot.choice(adata.n_obs, size=N_PLOT, replace=False)

umap_pca  = adata.obsm['X_umap'][plot_idx]
umap_scvi = adata.obsm['X_umap_scvi'][plot_idx]
obs_sub   = adata.obs.iloc[plot_idx]

def _scatter(ax, coords, c, cmap=None, title='', s=1, alpha=0.15, **kw):
    sc_p = ax.scatter(coords[:, 0], coords[:, 1], s=s, alpha=alpha,
                      c=c, cmap=cmap, rasterized=True, **kw)
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
    ax.set_title(title, fontsize=10)
    return sc_p

fig, axes = plt.subplots(4, 2, figsize=(14, 22))

# Row 1: Control vs perturbed
ctrl = (obs_sub['perturbation'] == 'control').values
for j, (coords, label) in enumerate([(umap_pca, 'PCA-UMAP'), (umap_scvi, 'scVI-UMAP')]):
    _scatter(axes[0, j], coords[~ctrl], c='steelblue', title=f'{label}: control vs. perturbed')
    _scatter(axes[0, j], coords[ctrl],  c='tomato', alpha=0.3)
    from matplotlib.patches import Patch
    axes[0, j].legend(handles=[
        Patch(color='tomato',    label='control'),
        Patch(color='steelblue', label='perturbed'),
    ], fontsize=8, markerscale=5)

# Row 2: UMI count (should be absent in scVI-UMAP)
col_umi_vals = np.log1p(obs_sub[col_umi].values) if col_umi in obs_sub.columns else np.zeros(N_PLOT)
for j, (coords, label) in enumerate([(umap_pca, 'PCA-UMAP'), (umap_scvi, 'scVI-UMAP')]):
    sc_p = _scatter(axes[1, j], coords, c=col_umi_vals, cmap='viridis',
                    title=f'{label}: coloured by log UMI count')
    plt.colorbar(sc_p, ax=axes[1, j], label='log1p(UMI)', shrink=0.7)

# Row 3: Perturbation cell tier
tier_colors = {'tier1_reliable': 'steelblue', 'tier2_caution': 'orange',
               'tier3_exclude': 'red', 'control': 'lightgray'}
tier_vals = obs_sub['pert_cell_tier'].map(tier_colors).fillna('gray').values
for j, (coords, label) in enumerate([(umap_pca, 'PCA-UMAP'), (umap_scvi, 'scVI-UMAP')]):
    _scatter(axes[2, j], coords, c=tier_vals, title=f'{label}: perturbation cell tier')
    axes[2, j].legend(
        handles=[Patch(color=v, label=k) for k, v in tier_colors.items()],
        fontsize=7, loc='lower right'
    )

# Row 4: Guide ambiguity
ambig_vals = obs_sub['guide_ambiguous'].map({True: 'tomato', False: 'steelblue'}).values
for j, (coords, label) in enumerate([(umap_pca, 'PCA-UMAP'), (umap_scvi, 'scVI-UMAP')]):
    _scatter(axes[3, j], coords, c=ambig_vals, title=f'{label}: guide ambiguity')
    axes[3, j].legend(
        handles=[Patch(color='tomato', label='ambiguous'), Patch(color='steelblue', label='clean')],
        fontsize=8
    )

plt.suptitle('PCA-UMAP vs. scVI-UMAP — ReplogleWeissman2022_rpe1', y=1.01, fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / '01b_pca_vs_scvi_umap.pdf', bbox_inches='tight')
plt.show()

<a id='pert-viz'></a>
## 9. Perturbation-level Visualizations

We highlight several well-characterised perturbations on both UMAPs to assess whether
the embedding captures known biology:

- **Essential genes** (expect few cells, tight clusters far from control)
- **Transcription factors** with known effects on RPE1 cell state
- **Controls** (should form a coherent cluster)

A good embedding will show perturbation groups occupying distinct regions of UMAP space
(reflecting their distinct transcriptional programs), while the scVI embedding should
produce tighter, more separated groups than PCA.

In [ ]:
# Select perturbations to highlight:
# Top 8 by E-distance-like proxy (cells far from control centroid in PCA space)
control_centroid = adata.obsm['X_scVI'][adata.obs['perturbation'] == 'control'].mean(axis=0)

# Mean scVI embedding per perturbation
pert_means = (
    pd.DataFrame(adata.obsm['X_scVI'], index=adata.obs_names)
    .assign(perturbation=adata.obs['perturbation'].values)
    .groupby('perturbation')
    .mean()
)

# Euclidean distance from control centroid
pert_dist = np.sqrt(((pert_means - control_centroid) ** 2).sum(axis=1))
pert_dist = pert_dist.drop('control', errors='ignore')

# Filter to tier 1 perturbations (enough cells for reliable centroid)
tier1_perts = set(adata.obs.loc[
    adata.obs['pert_cell_tier'] == 'tier1_reliable', 'perturbation'
].unique())
pert_dist_t1 = pert_dist[pert_dist.index.isin(tier1_perts)].sort_values(ascending=False)

top_perts = pert_dist_t1.head(8).index.tolist()
print(f'Top 8 perturbations by distance from control in scVI space:')
for i, p in enumerate(top_perts):
    n = (adata.obs['perturbation'] == p).sum()
    print(f'  {i+1}. {p:20s} ({n:,} cells, dist={pert_dist_t1[p]:.3f})')

# ---- Plot highlighted perturbations on both UMAPs -------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for j, (umap_key, label) in enumerate([('X_umap', 'PCA-UMAP'), ('X_umap_scvi', 'scVI-UMAP')]):
    ax = axes[j]
    coords = adata.obsm[umap_key]
    # Background: all cells in light gray
    ax.scatter(coords[:, 0], coords[:, 1],
              s=0.5, alpha=0.05, c='lightgray', rasterized=True)
    # Control cells
    ctrl_mask_ = adata.obs['perturbation'] == 'control'
    ax.scatter(coords[ctrl_mask_, 0], coords[ctrl_mask_, 1],
              s=2, alpha=0.15, c='black', label='control', rasterized=True)

    # Highlighted perturbations in distinct colours
    cmap_ = plt.cm.Set1
    for k, pert in enumerate(top_perts):
        mask = (adata.obs['perturbation'] == pert).values
        n_cells = mask.sum()
        ax.scatter(coords[mask, 0], coords[mask, 1],
                  s=5, alpha=0.7, c=[cmap_(k / len(top_perts))],
                  label=f'{pert} ({n_cells})', zorder=5)

    ax.set_title(f'{label}: top 8 perturbations', fontsize=11)
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.legend(fontsize=7, loc='lower right', markerscale=3,
             framealpha=0.9, ncol=1)

plt.suptitle('Highlighted perturbations — PCA vs. scVI embedding', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / '01b_highlighted_perturbations.pdf', bbox_inches='tight')
plt.show()

<a id='leiden'></a>
## 10. Leiden Clustering on scVI Embedding

Leiden clustering identifies groups of cells with similar transcriptional programs
in the scVI latent space. These clusters can reveal:

1. **Shared perturbation programs:** Multiple perturbations that converge on the same
   transcriptional state (e.g., different genes in the same pathway)
2. **Control subpopulations:** Heterogeneity within the control population (cell cycle,
   baseline variation)
3. **Outlier perturbations:** Perturbations with extreme effects that form isolated clusters

Leiden clustering operates on the k-NN graph (from scVI) and optimises a modularity
objective — it finds densely connected subgraphs. The `resolution` parameter controls
granularity: higher resolution → more clusters.

In [ ]:
# Run Leiden clustering on the scVI-based k-NN graph
sc.tl.leiden(
    adata,
    neighbors_key='scvi_neighbors',
    resolution=0.5,     # moderate resolution; adjust based on visual inspection
    key_added='leiden_scvi',
    random_state=42,
)

n_clusters = adata.obs['leiden_scvi'].nunique()
print(f'Leiden clustering (resolution=0.5): {n_clusters} clusters')
print()
print('Cells per cluster:')
print(adata.obs['leiden_scvi'].value_counts().sort_index().to_string())

# Visualise clusters on scVI-UMAP
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))

# Left: Leiden clusters
sc.pl.embedding(
    adata, basis='X_umap_scvi', color='leiden_scvi',
    ax=axes[0], show=False, title='scVI-UMAP: Leiden clusters',
    legend_loc='right margin', frameon=False, s=3,
)

# Right: top perturbations per cluster — which perturbations dominate each cluster?
# Compute the most common non-control perturbation per cluster
cluster_top_pert = (
    adata.obs[adata.obs['perturbation'] != 'control']
    .groupby('leiden_scvi')['perturbation']
    .agg(lambda x: x.value_counts().index[0])
)

cluster_summary = (
    adata.obs
    .groupby('leiden_scvi')
    .agg(
        n_cells=('leiden_scvi', 'count'),
        n_perts=('perturbation', 'nunique'),
        frac_ctrl=('perturbation', lambda x: (x == 'control').mean()),
    )
    .assign(top_perturbation=cluster_top_pert)
)
print('\nCluster summary:')
print(cluster_summary.to_string())

# Heatmap: cluster × perturbation enrichment (top 30 perturbations)
top30_perts = adata.obs.loc[
    adata.obs['perturbation'] != 'control', 'perturbation'
].value_counts().head(30).index.tolist()

crosstab = pd.crosstab(
    adata.obs['leiden_scvi'],
    adata.obs['perturbation'],
)[top30_perts]
# Normalise by column (fraction of each perturbation's cells in each cluster)
crosstab_norm = crosstab.div(crosstab.sum(axis=0), axis=1)

ax1 = axes[1]
sns.heatmap(crosstab_norm.T, cmap='viridis', ax=ax1,
            xticklabels=True, yticklabels=True, linewidths=0.3)
ax1.set_title('Perturbation enrichment per Leiden cluster\n(fraction of cells)', fontsize=10)
ax1.set_xlabel('Leiden cluster')
ax1.set_ylabel('Perturbation')
ax1.tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.savefig(FIG_DIR / '01b_leiden_scvi.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# <a id='save'></a>
# === 11. Save results ======================================================
# The main adata now contains:
#   adata.obsm['X_pca']        — 50-PC PCA embedding (from 01a)
#   adata.obsm['X_umap']       — PCA-based UMAP
#   adata.obsm['X_scVI']       — 10-D scVI latent space
#   adata.obsm['X_umap_scvi']  — scVI-based UMAP
#   adata.obs['leiden_scvi']   — Leiden clusters on scVI graph
#   adata.raw                  — raw counts

print(f'Saving to: {OUT_PATH}')
adata.write_h5ad(OUT_PATH, compression='gzip')
print(f'Saved locally ({OUT_PATH.stat().st_size / 1e9:.2f} GB).')

# Upload to S3
s3 = boto3.client('s3')
s3_key = f'processed/scperturb/{OUT_PATH.name}'
try:
    print(f'Uploading to s3://{S3_BUCKET}/{s3_key} ...')
    s3.upload_file(str(OUT_PATH), S3_BUCKET, s3_key)
    print('S3 upload complete.')
except Exception as e:
    print(f'S3 upload failed: {e}')
    print(f'File is available locally at {OUT_PATH}')

# Save scVI model weights to S3
try:
    import shutil
    model_tar = str(MODEL_DIR) + '.tar.gz'
    shutil.make_archive(str(MODEL_DIR), 'gztar', str(MODEL_DIR.parent), MODEL_DIR.name)
    s3.upload_file(model_tar, S3_BUCKET, f'results/model_checkpoints/scvi_model_rpe1.tar.gz')
    print('scVI model checkpoint uploaded to S3.')
except Exception as e:
    print(f'Model checkpoint S3 upload failed: {e}')

In [ ]:
# <a id='summary'></a>
print('=' * 65)
print('01b DIMENSIONALITY REDUCTION — SUMMARY')
print('=' * 65)
print()

checks = [
    ('QC AnnData loaded from 01a',                 PROC_PATH.exists()),
    ('PCA-UMAP computed',                          'X_umap' in adata.obsm),
    ('scVI model trained or loaded',               MODEL_DIR.exists()),
    ('scVI latent space (X_scVI) extracted',        'X_scVI' in adata.obsm),
    ('scVI-corrected UMAP computed',               'X_umap_scvi' in adata.obsm),
    ('Leiden clustering on scVI graph',            'leiden_scvi' in adata.obs.columns),
    ('Side-by-side comparison plotted',            True),
    ('Perturbation highlights plotted',            True),
    ('Processed AnnData saved',                    OUT_PATH.exists()),
]

all_pass = True
for check, result in checks:
    status = 'v' if result else 'X'
    print(f'  [{status}]  {check}')
    if not result:
        all_pass = False

print()
print(f'  Embeddings in adata.obsm:')
for key in sorted(adata.obsm.keys()):
    print(f'    {key:20s}  {adata.obsm[key].shape}')
print()
print(f'  scVI model: n_latent={N_LATENT}, n_hidden={N_HIDDEN}, n_layers={N_LAYERS}')
print(f'  Leiden clusters: {adata.obs["leiden_scvi"].nunique()}')
print()
print('Phase 2 dimred checkpoint:', 'PASSED' if all_pass else 'INCOMPLETE (see above)')
print()
print('Next step: 01c_differential_expression.ipynb — pseudobulk DE with PyDESeq2')

In [ ]:
from datetime import datetime
import subprocess

timestamp   = datetime.now().strftime('%Y%m%d_%H%M')
report_dir  = Path('../../results/reports')
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / f'01b_dimred_visualization_{timestamp}.html'

nb_path = (
    __vsc_ipynb_file__
    if '__vsc_ipynb_file__' in dir()
    else '01b_dimred_visualization.ipynb'
)
subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'html', '--output', str(report_path), nb_path],
    check=False,
)
print(f'Report saved to {report_path}')